## Unit 06. 線性聯立方程式之求解 — 課堂作業
- 課程名稱: 電腦在化工上之應用 (ChemE-3502)
- 學年度: 114下
- 上課時間: 每週一 16:10-19:20
-
- 指導教授: 莊曜禎 助理教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit06_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit06'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy import linalg

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
## I. 練習題：液體摻合問題（修改版）

某工廠有五個儲存槽，每一個槽中各含有不同體積率（%）的五種成分（A、B、C、D、E），各槽組成如下表：

| 槽號 | A (%) | B (%) | C (%) | D (%) | E (%) |
|------|-------|-------|-------|-------|-------|
| 1    | 55.9  | 7.9   | 16.3  | 11.5  | 8.4   |
| 2    | 20.1  | 52.2  | 11.2  | 9.8   | 6.7   |
| 3    | 17.6  | 12.2  | 46.0  | 14.3  | 9.9   |
| 4    | 18.3  | 13.8  | 11.7  | 47.1  | 9.1   |
| 5    | 19.5  | 18.2  | 21.5  | 10.6  | 30.2  |

現依顧客要求擬摻合成 **60 公升** 的產品，且各成分（A、B、C、D、E）體積率（%）分別為 **30、20、22、16、12**。在密度之改變可以忽略的假設情況下，試決定各槽所需用量（公升）。

令各槽所需之體積分別為 $V_1, V_2, V_3, V_4, V_5$ ，根據各成分的物料平衡可建立線性聯立方程組：

$$
\mathbf{A}\mathbf{x} = \mathbf{b}
$$

其中：

$$
\mathbf{A} = \frac{1}{100} \begin{bmatrix}
55.9 & 20.1 & 17.6 & 18.3 & 19.5 \\
7.9  & 52.2 & 12.2 & 13.8 & 18.2 \\
16.3 & 11.2 & 46.0 & 11.7 & 21.5 \\
11.5 & 9.8  & 14.3 & 47.1 & 10.6 \\
8.4  & 6.7  & 9.9  & 9.1  & 30.2
\end{bmatrix}, \quad
\mathbf{x} = \begin{bmatrix} V_1 \\ V_2 \\ V_3 \\ V_4 \\ V_5 \end{bmatrix}, \quad
\mathbf{b} = \begin{bmatrix} 0.30 \times 60 \\ 0.20 \times 60 \\ 0.22 \times 60 \\ 0.16 \times 60 \\ 0.12 \times 60 \end{bmatrix}
$$

In [ ]:
# ============================================================
# 已知數據 (請勿修改此儲存格)
# ============================================================

# 各槽各成分之體積率 (%) — 5x5 係數矩陣
tank_comp = np.array([
    [55.9, 20.1, 17.6, 18.3, 19.5],   # 成分 A
    [ 7.9, 52.2, 12.2, 13.8, 18.2],   # 成分 B
    [16.3, 11.2, 46.0, 11.7, 21.5],   # 成分 C
    [11.5,  9.8, 14.3, 47.1, 10.6],   # 成分 D
    [ 8.4,  6.7,  9.9,  9.1, 30.2],   # 成分 E
])

# 目標產品規格 (體積率 %) 與總量 (公升)
target_frac = np.array([30.0, 20.0, 22.0, 16.0, 12.0])  # %
total_vol   = 60.0  # 公升

# 建立係數矩陣 A 與右側向量 b
A = tank_comp / 100.0
b = target_frac / 100.0 * total_vol

print("係數矩陣 A:")
print(A)
print("\n右側向量 b (各成分目標量，公升):")
print(b)
print(f"\n方程組規模: {A.shape[0]} 個方程式，{A.shape[1]} 個未知數")

### **1.1 解的唯一性分析**
- 計算係數矩陣 $\mathbf{A}$ 的秩（rank）：`np.linalg.matrix_rank(A)`
- 計算擴增矩陣 $[\mathbf{A} \mid \mathbf{b}]$ 的秩
- 根據秩的判定準則，判斷此方程組屬於哪種情況（唯一解 / 無窮多解 / 無解）
- 計算行列式值 $\det(\mathbf{A})$ 並說明其物理意義

In [ ]:
# 請在此撰寫解的唯一性分析程式碼


### **1.2 求解方程組**
- 使用 `scipy.linalg.solve(A, b)` 求解各槽所需用量 $V_1, \ldots, V_5$ （公升）
- 印出每個槽的所需體積，並確認各數值均為正值（物理合理性）
- **問題**：若某槽的求解結果為負值，代表什麼物理意義？應如何處理？

In [ ]:
# 請在此撰寫求解程式碼


### **1.3 驗證解答**
- 計算並印出殘差向量： $\mathbf{r} = \mathbf{A}\mathbf{x} - \mathbf{b}$
- 計算殘差的 $L_2$ 範數：`np.linalg.norm(r)`，確認其非常接近 0
- 確認各槽用量總和等於 60 公升（整體物料平衡）
- 以長條圖（bar chart）繪製各槽所需用量，標示各槽編號與用量數值

In [ ]:
# 請在此撰寫驗證程式碼（含視覺化）


### **1.4 思考與討論**

請回答下列問題（以文字說明即可，無需程式碼）：

1. 若將目標產品組成改為 **A:20%, B:10%, C:10%, D:50%, E:10%**，方程組的解是否仍然有效？請說明判斷方式。
2. 五個槽的組成矩陣 $\mathbf{A}$ 每一列（row）的物理意義是什麼？
3. 若槽的數量少於成分的數量（例如只有 3 個槽，但有 5 種成分），方程組的類型會如何改變？

In [ ]:
# 可在此撰寫驗証或補充分析程式碼


---
## II. 練習題：蒸餾塔組之成分分析

某工廠利用以下的蒸餾塔組，分離對二甲苯（para-xylene）、苯乙烯（styrene）、甲苯（toluene）以及苯（benzene）等四成分。進料流量為 70 mol/hr，其中各成分莫耳分率分別為：對二甲苯 0.15、苯乙烯 0.25、甲苯 0.40、苯 0.20。

蒸餾塔組由三個蒸餾塔（Tower 1、Tower 2、Tower 3）串並聯組成，各出口流之已知組成如下：

| 出口流 |  對二甲苯 | 苯乙烯 | 甲苯 | 苯  |
|--------|---------|--------|------|-----|
| $D_2$  | 0.07    | 0.04   | 0.54 | 0.35 |
| $B_2$  | 0.18    | 0.24   | 0.42 | 0.16 |
| $D_3$  | 0.15    | 0.10   | 0.54 | 0.21 |
| $B_3$  | 0.24    | 0.65   | 0.10 | 0.01 |

其中 $D_2, B_2, D_3, B_3$ 分別為三個塔的塔頂及塔底出口流量（mol/hr）。

**(a)** 對各出口成分進行整體物料平衡，求解 $D_2, B_2, D_3, B_3$ 的流量（mol/hr）。

**(b)** 根據(a)的結果，計算第一個塔的塔頂 $D_1$ 及塔底 $B_1$ 各成分的莫耳分率組成。

（其中 $D_1 = D_2 + B_2$ ， $B_1 = D_3 + B_3$ ）

In [ ]:
# ============================================================
# 已知數據 (請勿修改此儲存格)
# ============================================================

# 各出口流之成分組成 (莫耳分率) — 各欄對應 D2, B2, D3, B3
#         D2     B2     D3     B3
comp = np.array([
    [0.07, 0.18, 0.15, 0.24],   # 對二甲苯
    [0.04, 0.24, 0.10, 0.65],   # 苯乙烯
    [0.54, 0.42, 0.54, 0.10],   # 甲苯
    [0.35, 0.16, 0.21, 0.01],   # 苯
])

# 進料條件
F_feed      = 70.0   # mol/hr
feed_frac   = np.array([0.15, 0.25, 0.40, 0.20])  # 各成分莫耳分率 [pX, sty, tol, ben]

# 整體各成分進料量（各成分之 b 向量）
b2 = F_feed * feed_frac  # mol/hr

# 係數矩陣即為 comp（各出口成分分率）
A2 = comp.copy()

print("係數矩陣 A (各出口成分分率):")
print(A2)
print("\n右側向量 b (各成分整體進料量, mol/hr):")
print(b2)

### **2.1 (a) 求解各出口流量**
- 使用 `scipy.linalg.solve(A2, b2)` 求解 $D_2, B_2, D_3, B_3$ （mol/hr）
- 驗證解的唯一性（rank 判定）
- 計算並印出殘差 $\|\mathbf{A}\mathbf{x} - \mathbf{b}\|$

In [ ]:
# 請在此撰寫 (a) 求解各出口流量的程式碼


### **2.2 (b) 計算 $D_1$ 與 $B_1$ 各成分組成**
- 由 (a) 的結果計算 $D_1 = D_2 + B_2$ 及 $B_1 = D_3 + B_3$ （mol/hr）
- 計算 $D_1$ 各成分莫耳分率：$x_i = (D_2 \cdot y_{D2,i} + B_2 \cdot y_{B2,i}) / D_1$
- 計算 $B_1$ 各成分莫耳分率：$\tilde{x}_i = (D_3 \cdot y_{D3,i} + B_3 \cdot y_{B3,i}) / B_1$
- 繪製堆疊長條圖（stacked bar chart），比較進料 $F$ 、 $D_1$ 、 $B_1$ 之各成分組成分布

In [ ]:
# 請在此撰寫 (b) 計算 D1, B1 成分組成及繪圖的程式碼


### **2.3 整體物料平衡驗證**
- 驗證：$D_1 + B_1 = F_{\text{feed}} = 70$ mol/hr
- 驗證：各成分整體流量守恆，即 $D_1 \cdot x_i + B_1 \cdot \tilde{x}_i = F_{\text{feed}} \cdot z_i$ （對每個成分 $i$ ）
- 以表格格式整理並印出：各股流的流量與各成分組成

In [ ]:
# 請在此撰寫整體物料平衡驗證程式碼


---
## III. 練習題：廢氣回收丙酮程序

下圖（示意）為從廢氣中回收丙酮的設備。廢氣由吸收塔底部進入，水自上方淋灑，含丙酮的水溶液再經閃蒸分離器（Flash Separator）回收丙酮。

**系統假設：**
1. 進入吸收塔的廢氣不含水，離開的廢氣含飽和水蒸氣
2. 閃蒸分離器的氣液平衡關係為：

$$
y = 20.5 \, x
$$

其中 $x$ 與 $y$ 分別為液相及氣相中丙酮的莫耳分率，且系統中 $x = 0.03$ 。

**已知條件：**
- 進廢氣流量 $F_i = 600 \text{ lb/hr}$ ，丙酮重量分率 8%
- 進水流量 $W = 500 \text{ lb/hr}$

**未知變數：**

| 變數 | 物理意義 |
|------|---------|
| $F_0$ | 處理後廢氣流量（lb/hr） |
| $V$   | 由閃蒸分離器離開之富含丙酮氣流流量（lb/hr） |
| $L$   | 由閃蒸分離器離開之廢液流量（lb/hr） |

**對全系統進行物料平衡，可建立下列三個方程式：**

空氣平衡：

$$
(1-0.08) F_i = (1-0.03) F_0
$$

丙酮平衡：

$$
0.08 F_i = y \, V + 0.03 \, L
$$

水平衡：

$$
W = 0.03 \, F_0 + (1-y) V + (1-x) L
$$

試求 $F_0, V, L$ 及丙酮的回收率。

In [ ]:
# ============================================================
# 已知數據 (請勿修改此儲存格)
# ============================================================

F_i = 600.0   # lb/hr  進廢氣流量
W   = 500.0   # lb/hr  進水流量

x = 0.03          # 液相丙酮莫耳分率 (閃蒸分離器出口液相)
y = 20.5 * x      # 氣相丙酮莫耳分率 (由平衡關係 y=20.5x 計算)

print(f"已知 x = {x}")
print(f"由平衡關係計算 y = 20.5 × {x} = {y}")
print(f"\n進廢氣流量 F_i = {F_i} lb/hr")
print(f"進水流量   W   = {W} lb/hr")

# 三個方程式整理為 A3 x3 = b3  (未知數: F0, V, L)
# 空氣: (1-0.03)*F0 + 0*V + 0*L = (1-0.08)*F_i
# 丙酮: 0*F0 + y*V + 0.03*L     = 0.08*F_i
# 水:   0.03*F0 + (1-y)*V + (1-x)*L = W

A3 = np.array([
    [(1 - 0.03), 0.0,    0.0      ],   # 空氣平衡
    [0.0,        y,       0.03     ],   # 丙酮平衡
    [0.03,       (1-y),  (1-x)    ],   # 水平衡
])

b3 = np.array([
    (1 - 0.08) * F_i,   # 空氣
    0.08 * F_i,         # 丙酮
    W,                  # 水
])

print("\n係數矩陣 A3:")
print(A3)
print("\n右側向量 b3:")
print(b3)

### **3.1 解的唯一性分析與求解**
- 計算 `rank(A3)` 與 `rank([A3|b3])`，判斷解的類型
- 使用 `scipy.linalg.solve(A3, b3)` 求解 $F_0, V, L$ （lb/hr）
- 印出求解結果並與教科書參考答案比較：$F_0 = 569.07, V = 54.82, L = 476.10$ （lb/hr）

In [ ]:
# 請在此撰寫解的唯一性分析與求解程式碼


### **3.2 計算丙酮回收率**

丙酮回收率定義為：從閃蒸分離器氣相中回收之丙酮量 除以 進廢氣中丙酮總量：

$$
\text{回收率 (\%)} = \frac{V \cdot y}{F_i \times 0.08} \times 100\%
$$

- 依據求解結果計算丙酮回收率
- 並與教科書參考答案 **70.24%** 比較
- 討論：若要提高回收率，可以調整哪些操作參數？

In [ ]:
# 請在此撰寫丙酮回收率計算程式碼


### **3.3 敏感度分析：進水量對回收率的影響**
- 改變進水量 $W$ ，範圍從 200 lb/hr 到 800 lb/hr（每步 50 lb/hr）
- 對每個 $W$ 值重新求解方程組，計算對應的丙酮回收率
- 繪製 $W$ 與丙酮回收率的關係曲線
- 在圖上標示原始操作點（$W = 500$ lb/hr）
- **結論**：進水量增加對回收率有何影響？

In [ ]:
# 請在此撰寫敏感度分析並繪圖的程式碼


---
## IV. 額外加分作業（自由繳交）

- 學生可自由選擇是否繳交加分作業（下週上課前完成 email 電子檔）
- 分數將加在最後總成績，加分幅度視完整度與分析深度而定
- 請務必自行完成！你可以查閱資料、詢問同學，但所有程式碼與分析說明必須是你自己理解後撰寫的
- **若系統自動比對偵測到抄襲，作業分數以 0 分計算**

---
### 加分題：三個異構體三角反應的穩態濃度分析

考慮三個異構體（isomer）A、B、C 之間的三角反應（triangle reaction），其反應動力學方程式為：

$$
\frac{d[A]}{dt} = -(k_1 + k_6)[A] + k_2[B] + k_5[C]
$$

$$
\frac{d[B]}{dt} = k_1[A] - (k_2 + k_3)[B] + k_4[C]
$$

$$
\frac{d[C]}{dt} = k_6[A] + k_3[B] - (k_4 + k_5)[C]
$$

其中 $[\cdot]$ 表示濃度。系統起始時只有 A 存在，且 $[A]_0 = 1.0$ （莫耳/公升）。

**提示**：在穩態時，所有微分均等於零（ $d[\cdot]/dt = 0$ ），可建立齊次線性方程組（homogeneous system）。由於此方程組必為線性相依（rank 必小於 3），需要加入**總量守恆約束**：

$$
[A] + [B] + [C] = [A]_0 = 1.0
$$

以取代其中一條方程式，使方程組有唯一解。

請分析以下**三種速率常數**情況下各成分的穩態濃度：

| 情況 | $k_1$ | $k_2$ | $k_3$ | $k_4$ | $k_5$ | $k_6$ |
|------|-------|-------|-------|-------|-------|-------|
| 1    | 1     | 0     | 1     | 0     | 1     | 0     |
| 2    | 1     | 1     | 1     | 1     | 1     | 1     |
| 3    | 2     | 1     | 1     | 1     | 1     | 1     |

**參考答案：**
- 情況一：[A]=0.333, [B]=0.333, [C]=0.333
- 情況二：[A]=0.333, [B]=0.333, [C]=0.333（加入總量守恆約束後，三成分均等分布）
- 情況三：[A]=0.250, [B]=0.417, [C]=0.333

### **4.1 情況一：$k_1 = k_3 = k_5 = 1, \; k_2 = k_4 = k_6 = 0$**

- 建立穩態齊次方程組（令 $d[\cdot]/dt = 0$ ），寫出對應的係數矩陣
- 分析 `rank` 說明為何需要加入總量守恆約束
- 用總量守恆取代第三條方程式，求解穩態濃度 $[A], [B], [C]$
- 驗證答案並說明物理意義

In [ ]:
# 請在此撰寫情況一的程式碼


### **4.2 情況二與三：比較分析**

- **情況二**：$k_1 = k_2 = k_3 = k_4 = k_5 = k_6 = 1$
  - 求解穩態濃度，並比較情況二與情況一的結果
  - 提示：當所有速率常數相等時，正逆反應完全對稱，穩態濃度均等分布，結果與情況一相同

- **情況三**：$k_1 = 2, \; k_2 = k_3 = k_4 = k_5 = k_6 = 1$
  - 求解穩態濃度，並與情況一、二比較

- **統整比較**：繪製三種情況的穩態濃度比較長條圖（三組並排），標示每種成分（A, B, C）的濃度

In [ ]:
# 請在此撰寫情況二、情況三的求解程式碼，及三種情況的比較圖


---
## 💭 思考題（非必答，可選擇回答）

1. `scipy.linalg.solve()` 與 `scipy.linalg.lstsq()` 的適用場合有何不同？
2. 什麼是「條件數（condition number）」？高條件數的矩陣在數值求解時會有什麼問題？
3. 線性方程組在化工製程分析中，最常見的應用情境是什麼？請舉出本課程以外的一個例子。
4. 若一個大型化工廠的全廠物料平衡有 500 個未知數和 500 個方程式，你認為應如何選擇求解策略（直接法 vs 迭代法，密集矩陣 vs 稀疏矩陣）？

---
# 想對老師說的話

（請在此填寫你對本次作業的感想、疑問或建議）